# Build, Shape, and Extend an OpenClaw Agent on AMD
### From local agent basics to skills, customization, and a live adaptive app

---

**Your setup today:**
- AMD Instinct MI300X GPU (192 GB memory)
- Qwen3.5-122B model served via SGLang
- OpenClaw agent framework
- This typing app as our demonstration vehicle

Everything runs on this machine. You don't need to configure anything — just run the cells in order.

---
## Section 0 — Your Environment

Each of you has a dedicated **AMD Instinct MI300X** GPU droplet (192 GB memory).  
The instructor has already done the heavy lifting before the session started:

```
✅ SGLang container running  →  Qwen3.5-122B loaded on port 8090
✅ OpenClaw installed        →  npm install -g openclaw@latest (Node 22)
```

Your job in this section: **verify the model is up, then configure and start OpenClaw**.

In [ ]:
# Check the SGLang model server is running and responding
import urllib.request, json

try:
    with urllib.request.urlopen("http://localhost:8090/v1/models") as r:
        models = json.loads(r.read())
    print("✅ Model server is up")
    for m in models.get("data", []):
        print(f"   Model: {m['id']}")
except Exception as e:
    print(f"❌ Model server not reachable: {e}")
    print("   Make sure the SGLang container is running (see instructor).")

In [ ]:
import json, subprocess

# ── Step 1: Verify SGLang is up ───────────────────────────────────────────────
import urllib.request

try:
    req = urllib.request.Request(
        "http://localhost:8090/v1/models",
        headers={"Authorization": "Bearer abc-123"}
    )
    with urllib.request.urlopen(req) as r:
        models = json.loads(r.read())
    print("✅ SGLang is running")
    for m in models.get("data", []):
        print(f"   Model: {m['id']}")
except Exception as e:
    print(f"❌ SGLang not reachable: {e}")
    print("   Tell the instructor — the container may need to be restarted.")

In [ ]:
import os, json, pathlib

# ── Step 2: Configure OpenClaw to point at the local SGLang server ────────────
# These are the same steps the setup script does non-interactively.

BASE_URL  = "http://localhost:8090/v1"
API_KEY   = "abc-123"
MODEL_ID  = "qwen3-5-122b"

# 2a — set the gateway mode and default model
subprocess.run(["openclaw", "config", "set", "gateway.mode", "local"], check=True)
subprocess.run(["openclaw", "config", "set", "agents.defaults.model", f"sglang/{MODEL_ID}"], check=True)

# 2b — write the provider block directly into openclaw.json
#      (openclaw config set can't build nested arrays, so we use Python)
config_path = pathlib.Path.home() / ".openclaw" / "openclaw.json"
with config_path.open() as f:
    cfg = json.load(f)

cfg.setdefault("models", {}).setdefault("providers", {})["sglang"] = {
    "baseUrl": BASE_URL,
    "apiKey":  API_KEY,
    "api":     "openai-completions",
    "models": [{
        "id":            MODEL_ID,
        "name":          MODEL_ID,
        "reasoning":     True,
        "input":         ["text"],
        "cost":          {"input": 0, "output": 0, "cacheRead": 0, "cacheWrite": 0},
        "contextWindow": 131072,
        "maxTokens":     8192,
    }]
}

with config_path.open("w") as f:
    json.dump(cfg, f, indent=2)

print("✅ OpenClaw configured")
print(f"   Provider : sglang")
print(f"   Base URL : {BASE_URL}")
print(f"   Model    : {MODEL_ID}")

---
## Section 1 — Start the Gateway and Talk to the Agent

The **gateway** is the OpenClaw process that sits between you and the model.  
We start it here, then send our first message.

In [ ]:
import subprocess, time, pathlib

# ── Step 3: Start the OpenClaw gateway ────────────────────────────────────────
# The gateway runs as a background process on port 18789.

log_file = pathlib.Path("/tmp/openclaw-gateway.log")

# Kill any existing gateway first so we get a clean start
subprocess.run(["pkill", "-9", "-f", "openclaw-gateway"], capture_output=True)
time.sleep(1)

proc = subprocess.Popen(
    ["openclaw", "gateway", "run", "--bind", "loopback", "--port", "18789", "--force"],
    stdout=log_file.open("w"),
    stderr=subprocess.STDOUT,
)

time.sleep(4)

if proc.poll() is None:
    print(f"✅ Gateway running  (PID {proc.pid})")
    print(f"   Logs: tail -f {log_file}")
else:
    print("❌ Gateway failed to start — check the log below:")
    print(log_file.read_text()[-800:])

In [ ]:
# ── Step 4: Send the first message ────────────────────────────────────────────
# If this prints a reply, the full stack is working:
#   Jupyter → openclaw chat → gateway → SGLang → Qwen3.5-122B

result = subprocess.run(
    ["openclaw", "chat", "Reply in one sentence: what model are you and what can you do?"],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

---
## Section 2 — How OpenClaw Is Structured

Before we start customizing, here is the mental model.

```
┌─────────────────────────────────────────────────────────────┐
│                        OpenClaw Stack                       │
│                                                             │
│  You ──► Gateway ──► Agent ──► Tools ──► World             │
│                         │                                   │
│                      Workspace                              │
│                   ┌─────┴──────┐                            │
│                 SOUL.md    USER.md        ← durable files   │
│                 AGENTS.md  SKILLS/        ← durable files   │
│                                                             │
│  Channels: WebChat · Slack · Teams · CLI                    │
│  ClawHub : shareable, installable skills                    │
└─────────────────────────────────────────────────────────────┘
```

| Concept | What it does |
|---|---|
| **Gateway** | Entry point — routes messages to the agent |
| **Agent** | Runs the loop: think → act → observe → repeat |
| **Workspace** | Where the agent lives and reads its context |
| **SOUL.md** | Defines the agent's personality and overall behavior |
| **USER.md** | Stores what the agent knows about you specifically |
| **Tools** | How the agent interacts with the world (files, browser, shell) |
| **Skills** | Reusable, packaged capabilities the agent can invoke |
| **Channels** | Different surfaces: same agent, different front-ends |
| **ClawHub** | Registry for sharing and installing skills |

> **Key idea:** the agent's behavior is shaped by *durable files in the workspace*, not just one-off prompts. Change the files, change the agent.

---
## Section 3 — Customize the Agent

We're going to make the agent more coach-like by editing two files: `SOUL.md` and `USER.md`.  
Watch how the agent's responses change after each edit.

In [ ]:
# See where the workspace lives
!openclaw workspace path

In [ ]:
# Show the current SOUL.md — this is what shapes agent personality
import subprocess
workspace = subprocess.check_output(["openclaw", "workspace", "path"]).decode().strip()
soul_path = f"{workspace}/SOUL.md"

with open(soul_path) as f:
    print(f.read())

In [ ]:
# Ask the agent something before customization — baseline behavior
!openclaw chat "I just typed 42 WPM with 88% accuracy. What should I do next?"

In [ ]:
# Append a coaching persona to SOUL.md
coaching_soul = """

## Coaching Mode
You are a focused typing coach. When users share typing results:
- Be direct and concise — no filler phrases
- Lead with the one most important thing to improve
- Give a specific, actionable next step
- Keep responses under 4 sentences
"""

with open(soul_path, "a") as f:
    f.write(coaching_soul)

print("✅ SOUL.md updated")

In [ ]:
# Ask the same question again — notice the difference
!openclaw chat "I just typed 42 WPM with 88% accuracy. What should I do next?"

In [ ]:
# Now add user-specific context to USER.md
user_path = f"{workspace}/USER.md"

user_context = """# About This User

- Developer who wants to improve typing speed for code, not prose
- Prefers direct, technical feedback — skip the encouragement
- Common weak words: 'their', 'receive', 'function', 'return'
- Goal: reach 70 WPM with 96%+ accuracy
"""

with open(user_path, "w") as f:
    f.write(user_context)

print("✅ USER.md written")

In [ ]:
# Same question — now the agent knows who it's talking to
!openclaw chat "I just typed 42 WPM with 88% accuracy. What should I do next?"

---
## Section 4 — Explore the Typing App

The typing app stores every session in a JSON file.  
Let's read it directly and give the agent something real to analyze.

In [ ]:
# Run a quick typing session to generate some data
# (if history already exists from a previous run, skip this)
import subprocess, os
history_path = os.path.expanduser("~/.open_type_faster_history.json")

if not os.path.exists(history_path):
    print("No history yet — run a session first:")
    print("  python3 main.py --words 15")
else:
    print(f"✅ History found at {history_path}")

In [ ]:
# Load and display session history as a table
import json, sys
sys.path.insert(0, ".")
import history as history_mod
from pathlib import Path

history_file = Path(history_path)
sessions = history_mod.load_sessions(history_file)

print(f"Total sessions: {len(sessions)}\n")
print(f"{'#':<4} {'Date':<12} {'Net WPM':<10} {'Accuracy':<10} {'Words':<8} {'Errors'}")
print("-" * 55)
for i, s in enumerate(sessions[:10], 1):
    print(f"{i:<4} {s['timestamp'][:10]:<12} {s['net_wpm']:<10} {s['accuracy']:<10} {s['word_count']:<8} {s['error_count']}")

In [ ]:
# Show the words this user struggles with most
word_errors = history_mod.load_word_errors(history_file)
ranked = sorted(word_errors.items(), key=lambda x: -x[1])[:10]

print("Top mistyped words:\n")
for word, count in ranked:
    bar = "█" * count
    print(f"  {word:<20} {bar} ({count})")

In [ ]:
# Feed the real data to the agent and ask for an analysis
summary = history_mod.summarise(history_file)

prompt = f"""Here are my typing stats:
- Sessions: {summary['total_sessions']}
- Best WPM: {summary['best_wpm']}
- Average WPM: {summary['avg_wpm']}
- Average accuracy: {summary['avg_accuracy']}%
- Most-missed words: {', '.join(summary['top_weak_words'])}

What is the single biggest pattern you see and what should I practice first?"""

print("Prompt sent to agent:")
print(prompt)
print("\n" + "─"*50 + "\n")

In [ ]:
import subprocess
result = subprocess.run(["openclaw", "chat", prompt], capture_output=True, text=True)
print(result.stdout)

---
## Section 5 — Build the Typing Coach Skill

Right now the analysis above is a one-off. A **skill** packages this into something reusable —  
the agent can invoke it any time, and you could share it on ClawHub.

We're going to create a skill called `typing-coach`.

In [ ]:
# Create the skill folder structure
import os
skill_dir = f"{workspace}/skills/typing-coach"
os.makedirs(skill_dir, exist_ok=True)
print(f"✅ Skill directory created: {skill_dir}")

In [ ]:
# Write the skill definition
skill_md = """# typing-coach

## Purpose
Analyze a user's typing session history and generate a targeted practice drill.

## When to use
Invoke this skill when the user shares typing results, asks what to practice,
or asks for a drill.

## Steps
1. Read the user's weak words and recent session stats (provided as input)
2. Identify the single most impactful pattern to address
3. Generate a 15-word practice sentence that targets that pattern
4. Give one concrete tip (max 2 sentences)

## Output format
```
PATTERN : <what you identified>
DRILL   : <15-word practice sentence>
TIP     : <one actionable tip>
```

## Rules
- The drill must be typeable — no punctuation except spaces
- The drill must heavily repeat the problematic words or letter combinations
- Never use filler phrases like 'Great job' or 'Keep it up'
"""

with open(f"{skill_dir}/skill.md", "w") as f:
    f.write(skill_md)

print("✅ typing-coach/skill.md written")
print()
print(skill_md)

In [ ]:
# Confirm OpenClaw can see the skill
!openclaw skills list

---
## Section 6 — Before vs After

Let's compare the generic response vs the skill-powered response on the same data.

In [ ]:
# Build the input from real history
weak_words = ", ".join(w for w, _ in ranked[:5])
latest = sessions[0] if sessions else {}

skill_input = f"""weak words: {weak_words}
last session: {latest.get('net_wpm', '?')} WPM, {latest.get('accuracy', '?')}% accuracy, {latest.get('error_count', '?')} errors"""

print("Input to skill:")
print(skill_input)

In [ ]:
# BEFORE — plain chat, no skill
print("=" * 50)
print("BEFORE (plain chat)")
print("=" * 50)
result = subprocess.run(
    ["openclaw", "chat", f"Based on this data, what should I practice? {skill_input}"],
    capture_output=True, text=True
)
print(result.stdout)

In [ ]:
# AFTER — invoke the typing-coach skill
print("=" * 50)
print("AFTER (typing-coach skill)")
print("=" * 50)
result = subprocess.run(
    ["openclaw", "skill", "typing-coach", skill_input],
    capture_output=True, text=True
)
print(result.stdout)

> **The app did not get smarter by accident. We added a capability.**

The skill is:
- Reusable — run it after every session
- Portable — move the `skills/typing-coach` folder to any OpenClaw workspace
- Shareable — publish it to ClawHub and anyone can `openclaw skill install typing-coach`

---
## Section 7 — ClawHub

What you built today stays local. ClawHub is how it becomes shareable.

```
Local skill  →  openclaw skill publish  →  ClawHub
Anyone       →  openclaw skill install typing-coach  →  their workspace
```

The same pattern applies to any capability you build — tools, skills, personas.
What you learn here scales directly to building things other people can use.

---
## Section 8 — Your Challenge

### Toolsmith Challenge

Build a skill or capability that makes your OpenClaw agent noticeably better at a visible task.

**Pick a track:**

| Track | What to build |
|---|---|
| **Typing Coach+** | Extend `typing-coach` to generate code-flavored drills (variable names, function calls) |
| **Weak Bigram Mode** | Identify the specific two-letter combinations the user fumbles and target them |
| **Streak Tracker** | Add a skill that tracks improvement across sessions and celebrates milestones |

**You have the full stack:**
- The history JSON with real per-session data
- The stats and history Python modules to call directly
- An agent with a custom SOUL and USER context
- A working skill as a template to copy and extend

---

*Need more GPU time? Members who share their projects publicly may apply for additional AMD Developer Cloud credits.*